# 92 — Multi-NR Transfer LGBM: Learn from all Nuclear Receptors

Train a single LGBM on ALL NR data (PPARg 4302 + FXR 3185 + RXRa 1364 + LXRa 1173 + VDR 523 + PXR 945 = ~11K compounds).
Use target-one-hot encoding → LGBM learns a shared NR representation.
Then fine-tune predictions on PXR-only data.

This is classical multi-task transfer: shared representation across related NRs.
The NR superfamily ligand-binding domains are structurally homologous (30-45% identity).

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
# Build idx_active / idx_inactive
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 0


In [4]:
from pxr.chem import to_inchikey

# Combine all NR data
chembl = pd.read_parquet(DATA_EXTERNAL/"chembl_nr_extended.parquet")
bdb    = pd.read_parquet(DATA_EXTERNAL/"bindingdb_nr_data.parquet")
nr_all = pd.concat([chembl, bdb], ignore_index=True)
nr_all = nr_all.dropna(subset=["smiles","pec50","target_name"])

NR_TARGETS = sorted(nr_all["target_name"].unique().tolist())
nr_all["target_idx"] = nr_all["target_name"].map({t:i for i,t in enumerate(NR_TARGETS)}).astype(int)
print(f"NR data: {len(nr_all):,} compounds across {len(NR_TARGETS)} targets: {NR_TARGETS}")

# Featurize all NR compounds: Morgan + RDKit + target one-hot
X_nr_comb = impute(combined(nr_all["smiles"].tolist()))
n_targets = len(NR_TARGETS)
target_oh = np.zeros((len(nr_all), n_targets), dtype=np.float32)
for i, tidx in enumerate(nr_all["target_idx"].values):
    target_oh[i, tidx] = 1.0
X_nr_full = np.hstack([X_nr_comb, target_oh])
y_nr = nr_all["pec50"].values.astype(np.float64)
print(f"Multi-NR feature matrix: {X_nr_full.shape}")


NR data: 17,186 compounds across 7 targets: ['FXR', 'LXRa', 'PPARa', 'PPARg', 'PXR', 'RXRa', 'VDR']


Multi-NR feature matrix: (17186, 2272)


In [5]:
# Train multi-NR LGBM (no CV — use all external data)
# Then fine-tune / stack with PXR-only model
LGBM_MNR = dict(n_estimators=500, num_leaves=64, learning_rate=0.05,
                min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
                reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)

print("Training multi-NR LGBM...", flush=True)
m_mnr = lgb.train(LGBM_MNR, lgb.Dataset(X_nr_full, label=y_nr),
                   callbacks=[lgb.log_evaluation(-1)])

# Get PXR target index
pxr_idx = NR_TARGETS.index("PXR") if "PXR" in NR_TARGETS else 0
print(f"PXR target index: {pxr_idx} (target: {NR_TARGETS[pxr_idx]})")

# Create PXR query features (combined + PXR one-hot)
pxr_oh_tr = np.zeros((len(tr), n_targets), dtype=np.float32)
pxr_oh_tr[:, pxr_idx] = 1.0
pxr_oh_te = np.zeros((len(te), n_targets), dtype=np.float32)
pxr_oh_te[:, pxr_idx] = 1.0
X_pxr_mnr_tr = np.hstack([X_tr, pxr_oh_tr])
X_pxr_mnr_te = np.hstack([X_te, pxr_oh_te])

# Multi-NR predictions for PXR train/test
pred_mnr_tr = m_mnr.predict(X_pxr_mnr_tr)
pred_mnr_te = m_mnr.predict(X_pxr_mnr_te)
print(f"Multi-NR preds on PXR train: mean={pred_mnr_tr.mean():.3f}  RAE={rae(y_tr, pred_mnr_tr):.4f}")


Training multi-NR LGBM...


PXR target index: 4 (target: PXR)
Multi-NR preds on PXR train: mean=5.051  RAE=1.0170


In [6]:
# Fine-tune: use multi-NR prediction as additional feature in PXR-only model
X_ft_tr = np.hstack([X_tr, pred_mnr_tr.reshape(-1,1)])
X_ft_te = np.hstack([X_te, pred_mnr_te.reshape(-1,1)])

oof = np.full(len(y_tr), np.nan)
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.train(LGBM, lgb.Dataset(X_ft_tr[tr_idx], label=y_tr[tr_idx]),
                  valid_sets=[lgb.Dataset(X_ft_tr[va_idx], label=y_tr[va_idx])],
                  callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof[va_idx] = m.predict(X_ft_tr[va_idx])
    print(f"  fold {fold+1}  RAE={rae(y_tr[va_idx], oof[va_idx]):.4f}", flush=True)

m_res = full_metrics(y_tr, oof, cliff_pairs, "multi_nr_transfer")
m_res_a = full_metrics(y_tr[active_mask], oof[active_mask], label="transfer [active]")

m_final = lgb.train(LGBM, lgb.Dataset(X_ft_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_preds = np.clip(m_final.predict(X_ft_te), y_tr.min()-0.5, y_tr.max()+0.5)
np.save(DATA_PROCESSED/"oof_multi_nr_transfer.npy", oof)
np.save(DATA_PROCESSED/"te_oof_multi_nr_transfer.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"92_multi_nr_transfer.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}  Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")


  fold 1  RAE=0.4952


  fold 2  RAE=0.5753


  fold 3  RAE=0.6020


  fold 4  RAE=0.5615


  fold 5  RAE=0.5945


  [multi_nr_transfer] RAE=0.5609 MAE=0.5103 R²=0.6041 r=0.7773 ρ=0.7310 τ=0.5392
  [transfer [active]] RAE=3.6490 MAE=0.7652 R²=-9.4527 r=0.0344 ρ=0.0695 τ=0.0476


Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\92_multi_nr_transfer.csv  Test: min=2.27 med=4.95 max=5.97
